### 1. Chargement et préparation

In [5]:
import os, re
from collections import Counter

data_path = "./data/txt"
files = sorted([f for f in os.listdir(data_path) if f.endswith(".txt")])
print("Nb de fichiers :", len(files))
print("Exemples :", files[:5])

STOP = {
    "les","des","une","un","le","la","de","du","au","aux","et","en","dans","sur",
    "que","qui","pour","par","plus","non","ne","pas","est","sont","cette","ces",
    "ces","ces","comme","avec","sans","ainsi","dont","entre","contre","où","il",
    "elle","elles","ils","on","nous","vous","se","sa","son","ses","leurs","leur",
    "d","l","j","qu","c","s","t","n","m"
}
token_re = re.compile(r"[a-zàâçéèêëîïôöùûüÿœæ-]{3,}", re.IGNORECASE)

def tokenize(txt:str):
    toks = [t.lower() for t in token_re.findall(txt)]
    return [t for t in toks if t not in STOP]

out_dir = "./output"
os.makedirs(out_dir, exist_ok=True)
print("Dossier sortie :", out_dir)


Nb de fichiers : 957
Exemples : ['KB_JB421_1950-01-04_01-00001.txt', 'KB_JB421_1950-01-04_01-00002.txt', 'KB_JB421_1950-01-04_01-00003.txt', 'KB_JB421_1950-01-04_01-00004.txt', 'KB_JB421_1950-01-04_01-00005.txt']
Dossier sortie : ./output


### 2. Les fréquences globales

In [6]:
all_tokens = []

for f in files:
    with open(os.path.join(data_path, f), encoding="utf-8", errors="replace") as fh:
        all_tokens.extend(tokenize(fh.read()))

freq = Counter(all_tokens)
top20 = freq.most_common(20)

print("Top 20 (token — fréquence) :")
for w, c in top20:
    print(f"{w:20s} {c:>7d}")

with open(os.path.join(out_dir, "top_tokens.csv"), "w", encoding="utf-8") as out:
    out.write("token,count\n")
    for w, c in top20:
        out.write(f"{w},{c}\n")



Top 20 (token — fréquence) :
ont                    12686
été                    12345
mais                   11104
deux                    9067
tout                    8379
était                   7817
avait                   7445
bien                    7356
fait                    6563
être                    6433
lui                     6210
après                   6023
tous                    6004
même                    5817
ans                     5097
encore                  5054
fut                     4962
très                    4908
sera                    4614
pays                    4539


### 3. Entités nommées 

In [7]:
import spacy
nlp = spacy.load("fr_core_news_sm")
print("Modèle spaCy OK")

sample_files = files[:300]

ent_counter = Counter()
label_counter = Counter()

for f in sample_files:
    with open(os.path.join(data_path, f), encoding="utf-8", errors="replace") as fh:
        doc = nlp(fh.read())
    for ent in doc.ents:
        ent_counter[(ent.text, ent.label_)] += 1
        label_counter[ent.label_] += 1

print("Répartition par label :")
for lab, c in label_counter.most_common():
    print(f"{lab:6s} -> {c}")

def top_by_label(label, n=15):
    return [(t,c) for (t,l),c in ent_counter.items() if l==label][:0]

for lab,_ in label_counter.most_common():
    top = [(t,c) for (t,l),c in ent_counter.items() if l==lab]
    top.sort(key=lambda x: -x[1])
    print(f"\nTop {min(15,len(top))} pour {lab}:")
    for t,c in top[:15]:
        print(f"{t[:50]:50s} ({lab}) – {c}")

with open(os.path.join(out_dir, "top_entities.csv"), "w", encoding="utf-8") as out:
    out.write("text,label,count\n")
    for (t,l),c in ent_counter.most_common():
        out.write(f"{t.replace(',',' ')},{l},{c}\n")


Modèle spaCy OK
Répartition par label :
LOC    -> 53017
PER    -> 50944
MISC   -> 29899
ORG    -> 15646

Top 15 pour LOC:
Bruxelles                                          (LOC) – 897
Belgique                                           (LOC) – 748
Arlon                                              (LOC) – 733
Gouvernement                                       (LOC) – 505
Paris                                              (LOC) – 456
I                                                  (LOC) – 448
Etat                                               (LOC) – 436
Liège                                              (LOC) – 393
Londres                                            (LOC) – 385
fr                                                 (LOC) – 379
Allemagne                                          (LOC) – 320
Luxembourg                                         (LOC) – 301
P                                                  (LOC) – 262
Namur                                              (LOC) – 

### 4. Clustering

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

docs = []
doc_names = []
for f in files[:1000]:
    with open(os.path.join(data_path, f), encoding="utf-8", errors="replace") as fh:
        docs.append(fh.read())
        doc_names.append(f)

print("Docs chargés :", len(docs))

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents=None,
    stop_words=list(STOP),
    ngram_range=(1,2),
    min_df=5,
    max_df=0.6
)
X = vectorizer.fit_transform(docs)
print("TF-IDF shape :", X.shape)

k = 5
km = KMeans(n_clusters=k, n_init="auto", random_state=42)
labels = km.fit_predict(X)

terms = vectorizer.get_feature_names_out()
order_centroids = km.cluster_centers_.argsort()[:, ::-1]

print("\nTop termes par cluster:")
for i in range(k):
    top_terms = [terms[ind] for ind in order_centroids[i, :15]]
    print(f"- Cluster {i}: {', '.join(top_terms)}")

from collections import defaultdict
groups = defaultdict(list)
for name, lab in zip(doc_names, labels):
    groups[lab].append(name)

print("\nExemples de fichiers par cluster:")
for i in range(k):
    print(f"Cluster {i}: {groups[i][:5]}")

with open(os.path.join(out_dir, "clusters.csv"), "w", encoding="utf-8") as out:
    out.write("file,cluster\n")
    for name, lab in zip(doc_names, labels):
        out.write(f"{name},{lab}\n")


Docs chargés : 957
TF-IDF shape : (957, 91455)

Top termes par cluster:
- Cluster 0: journal parlé, musique, parlé, parlante, 45, horloge parlante, horloge, œil monde, concert, coup œil, œil, orchestre, émission, radio, informations
- Cluster 1: ob, el, pv, 10e, cim, gr, cert, ind, belg, 89, lots, congo, hain, 88, transp
- Cluster 2: match, minute, équipe, classement, pts, champion, score, championnat, points, victoire, locaux, visiteurs, attaque, km, jeu
- Cluster 3: ministre, roi, parti, corée, politique, communistes, reuter, socialistes, etats, loi, europe, libéraux, allemagne, londres, spaak
- Cluster 4: kg, fr kg, marie, eau, votre, tu, travaux, bastogne, mme, enfants, film, mon, fille, frs, dc

Exemples de fichiers par cluster:
Cluster 0: ['KB_JB421_1950-03-04_01-00004.txt', 'KB_JB421_1950-03-05_01-00006.txt', 'KB_JB421_1950-03-06_01-00004.txt', 'KB_JB421_1950-03-07_01-00006.txt', 'KB_JB421_1950-03-08_01-00006.txt']
Cluster 1: ['KB_JB421_1950-01-04_01-00004.txt', 'KB_JB421_1950-0

### 5. Sentiment

In [9]:
from collections import Counter

POS_WORDS = {
    "bon","bonne","bien","excellent","excellente","super","positif","positive","succès",
    "heureux","heureuse","satisfait","satisfaite","favorable","gagner","victoire",
    "amélioration","progrès","meilleur","meilleure","fort","forte","stable","bénéfice"
}
NEG_WORDS = {
    "mauvais","mauvaise","mal","pire","négatif","négative","échec","crise","baisse",
    "perte","défaite","faible","instable","problème","problèmes","retard","diminution",
    "chômage","conflit","violence","scandale","dégradation","déficit"
}

def sentiment_score(tokens):
    if not tokens:
        return 0.0
    t = [t.lower() for t in tokens]
    pos = sum(1 for w in t if w in POS_WORDS)
    neg = sum(1 for w in t if w in NEG_WORDS)
    return (pos - neg) / max(1, len(t))

def label_from_score(s, th=0.02):
    if s > th:
        return "POS"
    if s < -th:
        return "NEG"
    return "NEU"

sample_names, preds = [], []
for f in files[:200]:  
    with open(os.path.join(data_path, f), encoding="utf-8", errors="replace") as fh:
        txt = fh.read(4000)  
    toks = tokenize(txt)
    s = sentiment_score(toks)
    preds.append({"file": f, "label": label_from_score(s), "score": round(s, 4)})
    sample_names.append(f)

agg = Counter([p["label"] for p in preds])
print("Répartition sentiment (échantillon 200 docs):", dict(agg))
for row in preds[:5]:
    print(row)


with open(os.path.join(out_dir, "sentiment_sample.csv"), "w", encoding="utf-8") as out:
    out.write("file,label,score\n")
    for p in preds:
        out.write(f"{p['file']},{p['label']},{p['score']}\n")
print("Fichier écrit :", os.path.join(out_dir, "sentiment_sample.csv"))


Répartition sentiment (échantillon 200 docs): {'NEU': 192, 'POS': 7, 'NEG': 1}
{'file': 'KB_JB421_1950-01-04_01-00001.txt', 'label': 'NEU', 'score': 0.0}
{'file': 'KB_JB421_1950-01-04_01-00002.txt', 'label': 'NEU', 'score': 0.0028}
{'file': 'KB_JB421_1950-01-04_01-00003.txt', 'label': 'NEU', 'score': 0.0}
{'file': 'KB_JB421_1950-01-04_01-00004.txt', 'label': 'NEU', 'score': 0.0027}
{'file': 'KB_JB421_1950-01-04_01-00005.txt', 'label': 'NEU', 'score': 0.0054}
Fichier écrit : ./output/sentiment_sample.csv


### 6. Word2Vec maison pour proximités lexicales

In [10]:
from gensim.models import Word2Vec

sentences = []
for f in files[:1000]:
    with open(os.path.join(data_path, f), encoding="utf-8", errors="replace") as fh:
        toks = tokenize(fh.read())
    sentences.append(toks)

w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=5, workers=4, epochs=10, sg=1)
print("Word2Vec entraîné.")

def show_similar(word, topn=10):
    if word in w2v.wv:
        print(f"\nMots proches de « {word} » :")
        for w,score in w2v.wv.most_similar(word, topn=topn):
            print(f"{w:20s} {score:.3f}")
    else:
        print(f"Mot absent du vocabulaire: {word}")

for seed in ["luxembourg","vote","gouvernement","parti","élections"]:
    show_similar(seed)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Word2Vec entraîné.

Mots proches de « luxembourg » :
avenir               0.781
samur                0.696
flatte               0.690
defend               0.678
brucelles            0.665
abonnant             0.658
dites-le             0.658
aidez-le             0.656
grand-duché          0.647
xembourg             0.642

Mots proches de « vote » :
nroiet               0.739
abstiendra           0.727
rejet                0.710
votèrent             0.710
affirmatif           0.707
recevabilité         0.707
nominal              0.705
abstention           0.702
émettra              0.697
investiture          0.693

Mots proches de « gouvernement » :
parlement            0.756
vernement            0.686
indonésien           0.682
inacceptables        0.679
gouver-              0.672
theotokis            0.671
heinemann            0.669
tripartite           0.668
boathby              0.666
interviendrait       0.661

Mots proches de « parti » :
socialiste           0.832
libéral          